In [65]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds
from collections import defaultdict, Counter

In [66]:
@njit
def split_top_continuous(tasks, priorities):
    """
    Sample a sequence of unique tasks of the highest priority ensuring that
    no task will have another instance with the priority level above the lowest priority in the sequence.
    Usecases: avoiding issues with "recommendations from future" when splitting test data by timestamp.
    """
    priority_queue = [(-max(priorities), len(priorities))]  # initialize typed
    priority_queue.pop()

    for idx, priority in enumerate(priorities):
        heapq.heappush(priority_queue, (-priority, idx))

    topseq = {}  # continuous sequence of top-priority tasks
    nonseq_idx = []  # top-priority tasks that interrupt continuous sequence

    unique_tasks = set(tasks)
    while unique_tasks:
        _, idx = heapq.heappop(priority_queue)
        task = tasks[idx]
        try:
            visited = topseq[task]
        except:
            unique_tasks.remove(task)
        else:
            nonseq_idx.append(visited)
        topseq[task] = idx

    topseq_idx = [idx for _, idx in topseq.items()]
    lowseq_idx = [idx for _, idx in priority_queue]  # all remaining tasks
    return topseq_idx, lowseq_idx, nonseq_idx


def to_numeric_array(series):
    if not is_numeric_dtype(series):
        if not hasattr(series, "cat"):
            series = series.astype("category")
        return series.cat.codes.values
    return series.values


# see polara

In [67]:
def earliest_last_out(data, userid="user_id", priority="timestamp", copy=False):
    """
    It helps avoiding "recommendations from future", when training set contains
    events that occur later than some events in the holdout and can therefore
    provide an oracle hint for the algorithm.
    """
    holdout_idx, observed_idx, future_idx = split_top_continuous(
        to_numeric_array(data[userid]), data[priority].values
    )

    observed = data.iloc[observed_idx]
    holdout = data.iloc[holdout_idx]
    future = data.iloc[future_idx]

    if copy:
        observed = observed.copy()
        holdout = holdout.copy()
        future = future.copy()

    return observed, holdout, future


# see polara

In [68]:
class MovieLens20MDataset(torch.utils.data.Dataset):
    """
    MovieLens 20M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path, sep=",", engine="c", header="infer"):
        self.data = pd.read_csv(
            dataset_path, sep=sep, engine=engine, header=header
        ).to_numpy()[:, :4]
        self.items = (
            self.data[:, :2].astype(np.int32) - 1
        )  # -1 because ID begins from 1
        self.targets = self.__preprocess_target(self.data[:, 2]).astype(np.float32)
        self.field_dims = np.max(self.items, axis=0) + 1
        self.user_field_idx = np.array((0,), dtype=np.int64)
        self.item_field_idx = np.array((1,), dtype=np.int64)

    def __len__(self):
        return self.targets.shape[0]

    def __getitem__(self, index):
        return self.items[index], self.targets[index]

    def __preprocess_target(self, target):
        target[target <= 0] = 0
        target[target > 0] = 1
        return target


class MovieLens1MDataset(MovieLens20MDataset):
    """
    MovieLens 1M Dataset

    Data preparation
        treat samples with a rating less than 3 as negative samples

    :param dataset_path: MovieLens dataset path

    Reference:
        https://grouplens.org/datasets/movielens
    """

    def __init__(self, dataset_path):
        super().__init__(dataset_path, sep="::", engine="python", header=None)

In [69]:
dataset_path = "data/MovieLens1M/ratings.dat"
dataset = MovieLens1MDataset(dataset_path)

In [70]:
user_num = dataset.field_dims[0]
item_num = dataset.field_dims[1]
print("Number of users: ", user_num, ", Number of items: ", item_num)

Number of users:  6040 , Number of items:  3952


In [71]:
columns_name = ["user_id", "item_id", "rating", "timestamp"]
df = pd.DataFrame(dataset.data, columns=columns_name)
df

,user_id,item_id,rating,timestamp
0,1,1193,1,978300760
1,1,661,1,978302109
2,1,914,1,978301968
3,1,3408,1,978300275
4,1,2355,1,978824291
...,...,...,...,...
1000204,6040,1091,1,956716541
1000205,6040,1094,1,956704887
1000206,6040,562,1,956704746
1000207,6040,1096,1,956715648


### Folding IN

1) Разбиваем earliest-last на test - это holdout и train - это всё остальное. т.е. old_train+new_train 

Для этого используем первое разбиение по какому-то timestamp тем самым делаем old_train

2) Делаем разбиение по новому timestamp. Тогда получится разбить на val и полноценный train.

3) Те пользователи, которые попали в test или val, становятся тестовыми пользователями. 

4) Для них запускаем warmstart - они warm train

In [124]:
df_sorted = df.sort_values(by="timestamp")

In [125]:
stab_size = int(len(df_sorted) * 0.95)
stab_train = df_sorted.head(stab_size)
all_df = stab_train.tail(len(df_sorted) - stab_size)
observed, holdout, future = earliest_last_out(all_df)


test = holdout

In [126]:
print("Test length:", len(test))
print("Test users:", len(test["user_id"].unique()))

Test length: 948
Test users: 948


In [127]:
train_val = pd.concat([stab_train, observed], ignore_index=True)
stab_size2 = int(len(train_val) * 0.995)
stab_train2 = train_val.head(stab_size)
all_df = stab_train.tail(len(train_val) - stab_size2)
observed2, holdout2, future2 = earliest_last_out(all_df)


val = holdout2

In [128]:
print(len(future2))

4475


In [129]:
print("Valid length:", len(val))
print("Valid users:", len(val["user_id"].unique()))
print(
    "Test and Val users:",
    len(np.intersect1d(test["user_id"].unique(), val["user_id"].unique())),
)

Valid length: 276
Valid users: 276
Test and Val users: 276


In [130]:
test_val_users = np.union1d(test["user_id"].unique(), val["user_id"].unique())

Train without leaving test_val users

In [131]:
train = pd.concat([stab_train2, observed2], ignore_index=True)

In [132]:
print("Train length:", len(train))
print("Train users:", len(train["user_id"].unique()))

Train length: 950200
Train users: 6036


In [133]:
filtered_train = train[~train["user_id"].isin(test_val_users)]

In [134]:
print("Train length:", len(filtered_train))
print("Train users:", len(filtered_train["user_id"].unique()))

Train length: 676047
Train users: 5088


In [135]:
warm_train = train[train["user_id"].isin(test_val_users)]

In [136]:
print("Warm start train length:", len(warm_train))
print("Warm start train users:", len(warm_train["user_id"].unique()))

Warm start train length: 274153
Warm start train users: 948


Теперь переделаем датасеты в txt файлы, где для каждого пользователя сохранена последовательность его items